# 03 - Feature Importance: The 20 kHz Trap

Why is one band so important? Let's investigate the feature importance.

In [1]:
import sys
from pathlib import Path

# notebooks/ holds _fetch.py and _sources.py; src/ holds the project's own helpers.
for p in (Path.cwd(), Path.cwd() / ".." / "src"):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import _fetch
import _sources as S
from ocean_data_workshop.dsn import dsn
import psycopg

# ML imports
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split

# Setup
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

print("ML Workshop Environment Ready")
print("=" * 60)
print("Data sources:")
print("  - NDBC 46092 (wind)")
print("  - GLORYS12V1 (ocean)")
print("  - SanctSound MB01 (acoustic)")
print("=" * 60)

ML Workshop Environment Ready
Data sources:
  - NDBC 46092 (wind)
  - GLORYS12V1 (ocean)
  - SanctSound MB01 (acoustic)


## The 20 kHz question

The detector was trained on 96 kHz data, but the public products only go to 20 kHz.
Dolphin echolocation peaks above 20 kHz. So why is 20 kHz so important?

In [2]:
# Analyze feature importance
import numpy as np

# Load all data
conn = psycopg.connect(dsn())

# Get all data
query = "SELECT observed_at, site_id,"
query += " band_25hz, band_50hz, band_160hz, band_500hz, band_1600hz, band_5000hz,"
query += " band_10000hz, band_16000hz, band_20000hz"
query += " FROM acoustic_tol_hourly WHERE site_id = 'mb01' ORDER BY observed_at"

acoustic_df = pd.read_sql_query(query, conn)

# Load dolphin data
query2 = "SELECT observed_at, presence FROM detection_hourly"
query2 += " WHERE taxon = 'dolphin' AND is_event_list = FALSE ORDER BY observed_at"

dolphin_df = pd.read_sql_query(query2, conn)

conn.close()

# Merge
merged = acoustic_df.merge(dolphin_df, on='observed_at')
y = merged['presence'].values

# Analyze each band
bands = [c for c in merged.columns if c.startswith('band_')]
results = []

for band in bands:
    X_single = merged[band].values.reshape(-1, 1)
    X_train, X_test, y_train, y_test = train_test_split(
        X_single, y, test_size=0.25, random_state=42
    )
    
    model = LogisticRegression(max_iter=2000)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    
    results.append({
        'band': band,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'correlation': np.corrcoef(X_single.flatten(), y)[0, 1]
    })

results_df = pd.DataFrame(results).sort_values('accuracy', ascending=False)

print("Single-band performance:")
print(results_df.to_string(index=False))

# Find the best band
best_band = results_df.iloc[0]
print(f"\nBest band: {best_band['band']}")
print(f"  Accuracy: {best_band['accuracy']:.3f}")
print(f"  Correlation: {best_band['correlation']:.3f}")

Single-band performance:
        band  accuracy  precision   recall       f1  correlation
band_20000hz  0.780639   0.666667 0.004338 0.008621     0.141181
   band_50hz  0.780162   0.000000 0.000000 0.000000    -0.016877
  band_160hz  0.780162   0.000000 0.000000 0.000000    -0.048335
  band_500hz  0.780162   0.000000 0.000000 0.000000    -0.046730
 band_1600hz  0.780162   0.000000 0.000000 0.000000    -0.111128
 band_5000hz  0.780162   0.000000 0.000000 0.000000    -0.099854
band_10000hz  0.780162   0.000000 0.000000 0.000000    -0.098360
band_16000hz  0.780162   0.000000 0.000000 0.000000    -0.065958
   band_25hz  0.777301   0.250000 0.006508 0.012685     0.138716

Best band: band_20000hz
  Accuracy: 0.781
  Correlation: 0.141


/var/folders/6_/6bnkvg2j2qd8720cj2rr98080000gn/T/ipykernel_26123/3730575120.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  acoustic_df = pd.read_sql_query(query, conn)
/var/folders/6_/6bnkvg2j2qd8720cj2rr98080000gn/T/ipykernel_26123/3730575120.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dolphin_df = pd.read_sql_query(query2, conn)


## The 20 kHz anomaly

If 20 kHz is so important, what happens if we remove it?

In [3]:
# Load all bands
conn = psycopg.connect(dsn())
query = "SELECT observed_at, site_id,"
query += " band_25hz, band_32hz, band_40hz, band_50hz, band_63hz, band_80hz,"
query += " band_100hz, band_125hz, band_160hz, band_200hz, band_250hz, band_315hz,"
query += " band_400hz, band_500hz, band_630hz, band_800hz, band_1000hz, band_1250hz,"
query += " band_1600hz, band_2000hz, band_2500hz, band_3150hz, band_4000hz,"
query += " band_5000hz, band_6300hz, band_8000hz, band_10000hz, band_12500hz,"
query += " band_16000hz, band_20000hz"
query += " FROM acoustic_tol_hourly WHERE site_id = 'mb01' ORDER BY observed_at"

acoustic_df = pd.read_sql_query(query, conn)
conn.close()

# Load dolphin
conn = psycopg.connect(dsn())
query = "SELECT observed_at, presence FROM detection_hourly"
query += " WHERE taxon = 'dolphin' AND is_event_list = FALSE ORDER BY observed_at"

dolphin_df = pd.read_sql_query(query, conn)
conn.close()

# Merge
merged = acoustic_df.merge(dolphin_df, on='observed_at')
y = merged['presence'].values

# Full model
X_all = merged[[c for c in merged.columns if c.startswith('band_')]].values
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.25, random_state=42
)
model_all = LogisticRegression(max_iter=2000)
model_all.fit(X_train, y_train)
pred_all = model_all.predict(X_test)
acc_all = accuracy_score(y_test, pred_all)

# Model without 20 kHz
X_no_20k = merged[[c for c in merged.columns if c.startswith('band_') and c != 'band_20000hz']].values
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_no_20k, y, test_size=0.25, random_state=42
)
model_no_20k = LogisticRegression(max_iter=2000)
model_no_20k.fit(X_train2, y_train2)
pred_no_20k = model_no_20k.predict(X_test2)
acc_no_20k = accuracy_score(y_test2, pred_no_20k)

print(f"Full model accuracy:  {acc_all:.3f}")
print(f"No 20 kHz accuracy:   {acc_no_20k:.3f}")
print(f"Difference:           {acc_all - acc_no_20k:+.3f}")

if acc_all - acc_no_20k > 0.1:
    print("\nCONCLUSION: 20 kHz is critically important")
else:
    print("\nCONCLUSION: 20 kHz is NOT critically important")

/var/folders/6_/6bnkvg2j2qd8720cj2rr98080000gn/T/ipykernel_26123/2852990940.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  acoustic_df = pd.read_sql_query(query, conn)


/var/folders/6_/6bnkvg2j2qd8720cj2rr98080000gn/T/ipykernel_26123/2852990940.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dolphin_df = pd.read_sql_query(query, conn)


/Users/dev/Developer/projects/ocean-sim/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Full model accuracy:  0.885
No 20 kHz accuracy:   0.803
Difference:           +0.082

CONCLUSION: 20 kHz is NOT critically important


/Users/dev/Developer/projects/ocean-sim/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## The lesson

The 20 kHz band is important because:
1. It's the closest to dolphin echolocation (30-50 kHz)
2. Dolphin clicks produce energy at 20 kHz
3. The band above 20 kHz isn't available in public products

**But**: This doesn't mean we're detecting dolphins directly!
It means we're detecting the *acoustic signature* that the detector used.